[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlog7997/rag-playbook/blob/master/learn/03-irrelevant-results/notebook.ipynb)

# Chapter 3 — Irrelevant Results (Right File, Wrong Section)

## The Real Story

After fixing retrieval with hybrid search, we noticed a new problem. Results were technically from the right file, but the **top-ranked chunk was always the introduction or overview section** — not the specific answer.

The introduction sections of our healthcare platform docs mentioned every topic covered in the file. "Authentication," "SSO," "two-factor," "session management" — all appeared in the first paragraph. So for almost any query about authentication, the intro scored highest. The actual detailed instructions were buried at rank 3, 4, or 5.

Users were getting answers like: *"NovaDash supports two-factor authentication via TOTP and SMS fallback, and you can configure this in the security settings."* Accurate but useless — no steps, no instructions, just the intro paragraph repeated.

**Reranking was the fix.**

---

## The Two-Stage Funnel

Think of retrieval + reranking as a two-stage funnel:

```
All 80+ chunks
      │
      ▼ Stage 1: Retrieval (fast, approximate)
      │  Hybrid search → top 15 candidates
      │  Goal: don't miss the right answer (high recall)
      │
      ▼ Stage 2: Reranking (slower, precise)
         Reranker scores each (query, chunk) pair directly
         Goal: move the best answer to #1 (high precision)
```

Retrieval is optimized for **recall** (cast a wide net). Reranking is optimized for **precision** (rank the net's contents correctly).

## What You'll Learn
- Cross-encoder reranking: before vs after comparison
- LLM-as-reranker with scores and reasons
- Contextual embeddings at ingest time
- Trade-off comparison table

## Setup

In [ ]:
# Install required packages — cross-encoder is part of sentence-transformers
!pip install anthropic==0.49.0 chromadb sentence-transformers rank-bm25 keyring -q

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

import keyring
# keyring.set_password("Anthorpic", "API_KEY", "your-key-here")  # run once
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
LEARN_DIR = REPO_ROOT / 'learn'
sys.path.insert(0, str(LEARN_DIR))

print(f"Repo root: {REPO_ROOT}")
print("Setup complete!")

## Test Queries for Reranking

These queries demonstrate the "intro section dominates" problem. The intro of each doc mentions all topics, so it scores highly for any query about that file.

In [ ]:
# Queries chosen to show the intro-bias problem
TEST_QUERIES = [
    {
        "query": "How do I set up two-factor authentication?",
        "expected_source": "authentication.md",
        "expected_section": "Two-Factor Authentication",
        "note": "Intro section mentions 2FA; specific 2FA section has setup steps",
    },
    {
        "query": "What webhook events does NovaDash support?",
        "expected_source": "notifications.md",
        "expected_section": "Webhook",
        "note": "Notifications intro mentions webhooks; specific webhook section has event types",
    },
    {
        "query": "How do I import users from a CSV file?",
        "expected_source": "user-management.md",
        "expected_section": "Bulk User Import",
        "note": "User management intro is broad; CSV import section is highly specific",
    },
]

print("Test queries loaded:")
for i, q in enumerate(TEST_QUERIES, 1):
    print(f"  [{i}] {q['query']}")
    print(f"       Expected: {q['expected_source']} > {q['expected_section']}")
    print(f"       Note: {q['note']}")

## Approach 1: Cross-Encoder Reranker

A **cross-encoder** takes a (query, document) pair as a single input and outputs a relevance score. Unlike bi-encoders (which embed query and doc separately), cross-encoders let the two texts "see" each other — enabling fine-grained comparison.

**Model:** `cross-encoder/ms-marco-MiniLM-L-12-v2`
- Trained on the MS MARCO web search relevance dataset
- Runs **locally** — no API call, no cost
- Speed: ~50ms per (query, chunk) pair on CPU. For 15 chunks, ~750ms total.
- Limitation: trained on web queries, may misunderstand SaaS domain terminology

Let's first load the model, then run the two-stage pipeline:

In [ ]:
from sentence_transformers import CrossEncoder

# Download on first run (~67 MB)
print("Loading cross-encoder model (ms-marco-MiniLM-L-12-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-12-v2")
print("Model loaded.\n")


def rerank_with_cross_encoder(query: str, hits: list) -> list:
    """
    Score each (query, chunk) pair using the cross-encoder.
    Returns hits sorted by cross-encoder score (descending).
    """
    # Feed each (query, text) pair to the cross-encoder
    pairs = [(query, hit["text"]) for hit in hits]
    scores = reranker.predict(pairs)

    reranked = []
    for hit, score in zip(hits, scores):
        reranked.append({**hit, "ce_score": float(score)})

    # Sort by cross-encoder score (higher = more relevant)
    reranked.sort(key=lambda x: x["ce_score"], reverse=True)
    return reranked


print("rerank_with_cross_encoder() function defined.")

In [ ]:
from utils import hybrid_search

FETCH_K = 15  # Stage 1: fetch 15 candidates, then rerank to top 5

print("=" * 70)
print("  Cross-Encoder Reranking: Before vs After")
print("=" * 70)

for item in TEST_QUERIES:
    query = item["query"]
    print(f"\n\nQuery: {query}")
    print(f"Note:  {item['note']}")
    print("-" * 70)

    # Stage 1: Retrieve wide net of candidates
    candidates = hybrid_search(query, top_k=FETCH_K, fetch_k=FETCH_K)
    print(f"Stage 1: Hybrid retrieval → {len(candidates)} candidates")

    # Stage 2: Cross-encoder reranking
    reranked = rerank_with_cross_encoder(query, candidates)

    # Show top 5 BEFORE reranking
    print("\nBEFORE reranking (hybrid order):")
    for i, hit in enumerate(candidates[:5], 1):
        is_target = (
            item["expected_source"] in hit["source"]
            and item["expected_section"].lower() in hit["section"].lower()
        )
        marker = ">>> TARGET" if is_target else "          "
        print(f"  [{i}] {marker} {hit['source']} > {hit['section']}")

    # Show top 5 AFTER reranking
    print("\nAFTER cross-encoder reranking:")
    for i, hit in enumerate(reranked[:5], 1):
        is_target = (
            item["expected_source"] in hit["source"]
            and item["expected_section"].lower() in hit["section"].lower()
        )
        marker = ">>> TARGET" if is_target else "          "
        print(f"  [{i}] {marker} {hit['source']} > {hit['section']}  "
              f"(CE score: {hit['ce_score']:.3f})")

    # Track rank improvement
    before_rank = next(
        (i + 1 for i, h in enumerate(candidates)
         if item["expected_source"] in h["source"]
         and item["expected_section"].lower() in h["section"].lower()), -1
    )
    after_rank = next(
        (i + 1 for i, h in enumerate(reranked)
         if item["expected_source"] in h["source"]
         and item["expected_section"].lower() in h["section"].lower()), -1
    )
    print(f"\n  Target section rank: {before_rank} >>> {after_rank} after reranking")

The target section reliably moves from rank 3+ to rank 1 after cross-encoder reranking.

## Approach 2: LLM-as-Reranker (Claude Haiku)

Instead of a fine-tuned cross-encoder, we ask **Claude Haiku** to score each chunk's relevance. The LLM understands domain-specific SaaS terminology that a web-search-trained model might miss.

**For each candidate chunk, we ask:**
```
"On a scale of 1-10, how relevant is this documentation chunk to the query?
Return JSON: {\"score\": 7, \"reason\": \"Contains step-by-step setup...\"}" 
```

**Pros:** Domain-aware, can reason about answer quality, provides explanations

**Cons:** ~15 API calls per query, adds 3-5 seconds of latency, costs more

In [ ]:
import json
import re
from utils import hybrid_search, ask_claude

RERANK_SYSTEM = """You are a relevance scoring assistant for a documentation search system.
You will receive a user query and a documentation chunk. Score how relevant the chunk is
to answering the query.

Return ONLY a JSON object in this exact format:
{"score": <integer 1-10>, "reason": "<one sentence explanation>"}

Scoring guide:
  10 = Chunk directly and completely answers the query with specific details or steps
   8 = Chunk is highly relevant and contains most of the needed information
   6 = Chunk is related to the topic but doesn't fully answer the query
   4 = Chunk is from the right domain but only tangentially related
   2 = Chunk mentions the topic in passing but is not useful for this query
   1 = Chunk is unrelated to the query"""


def score_chunk_with_llm(query: str, chunk_text: str) -> dict:
    """Ask Claude Haiku to score a single (query, chunk) pair."""
    user_prompt = f"""Query: {query}

Documentation chunk:
{chunk_text[:800]}

Score this chunk's relevance to the query."""

    response = ask_claude(
        system=RERANK_SYSTEM,
        user=user_prompt,
        model="claude-haiku-4-5-20251001",
    )

    try:
        clean = re.sub(r"```json\s*|\s*```", "", response).strip()
        result = json.loads(clean)
        return {"score": int(result["score"]), "reason": result.get("reason", "")}
    except (json.JSONDecodeError, KeyError, ValueError):
        match = re.search(r'"score"\s*:\s*(\d+)', response)
        score = int(match.group(1)) if match else 5
        return {"score": score, "reason": "parse fallback"}


def rerank_with_llm(query: str, hits: list) -> list:
    """Score all candidates with Claude Haiku and re-sort by score."""
    print(f"  Scoring {len(hits)} chunks with Claude Haiku...")
    reranked = []
    for i, hit in enumerate(hits):
        print(f"    [{i + 1}/{len(hits)}] {hit['source']} > {hit['section'][:40]}", end="", flush=True)
        score_result = score_chunk_with_llm(query, hit["text"])
        print(f" > score: {score_result['score']}")
        reranked.append({
            **hit,
            "llm_score": score_result["score"],
            "llm_reason": score_result["reason"],
        })

    reranked.sort(key=lambda x: x["llm_score"], reverse=True)
    return reranked


print("LLM reranker functions defined.")

In [ ]:
# Run LLM reranking on the first query (running all 3 would be slow)
# Note: This makes ~15 API calls to Claude Haiku
query_to_test = TEST_QUERIES[0]
query = query_to_test["query"]

print(f"Query: {query}")
print("=" * 70)

candidates = hybrid_search(query, top_k=FETCH_K, fetch_k=FETCH_K)
print(f"Retrieved {len(candidates)} candidates via hybrid search.\n")

llm_reranked = rerank_with_llm(query, candidates)

print("\nLLM Reranker — Top 5 results with scores and reasons:")
for i, hit in enumerate(llm_reranked[:5], 1):
    is_target = (
        query_to_test["expected_source"] in hit["source"]
        and query_to_test["expected_section"].lower() in hit["section"].lower()
    )
    marker = ">>> TARGET" if is_target else "          "
    print(f"  [{i}] {marker} {hit['source']} > {hit['section']}")
    print(f"       Score: {hit['llm_score']}/10 — {hit['llm_reason']}")

## Approach 3: Contextual Embeddings at Ingest Time

Instead of fixing the ranking *after* retrieval, we can fix the **embedding quality** at index time.

**The problem:** A chunk like `## Two-Factor Authentication\nSetup:\n1. Go to Profile...` has a poor embedding because it's instruction-heavy and keyword-sparse. The intro section embeds better for queries like "how does 2FA work" because it has a rich overview.

**The fix:** Prepend a context sentence (generated by LLM) that explicitly describes what the chunk is about:

```
Original chunk:
  "## Two-Factor Authentication\nNovaDash supports two 2FA methods..."

Enriched chunk:
  "This section explains how to enable two-factor authentication in NovaDash,
  covering TOTP setup with authenticator apps and SMS fallback configuration.
  ## Two-Factor Authentication\nNovaDash supports two 2FA methods..."
```

**Cost:** One Claude Haiku call per chunk at ingest time — ~$0.05 for 80 chunks. Done once.

In [ ]:
import time
from utils import load_bm25, get_embedding_model, ask_claude
import chromadb

DATA_DIR = REPO_ROOT / "data"
CHROMA_PATH = str(DATA_DIR / "chromadb")
CONTEXTUAL_COLLECTION = "novadash_contextual"

CONTEXT_SYSTEM = """You are a documentation indexing assistant. Your job is to write a
concise 1-2 sentence description of a documentation chunk that makes it easy to find
via search.

The description MUST:
1. State what topic or feature this section covers
2. State what question(s) this section answers for a user

Return ONLY the description — no JSON, no labels, no preamble. Just the 1-2 sentences."""


def generate_context_description(chunk_text: str, source: str, section: str) -> str:
    """Ask Claude Haiku to describe what a chunk is about and what question it answers."""
    user_prompt = f"""Documentation file: {source}
Section: {section}

Content:
{chunk_text[:600]}

Write a 1-2 sentence description of this section: what it covers and what user question it answers."""

    return ask_claude(
        system=CONTEXT_SYSTEM,
        user=user_prompt,
        model="claude-haiku-4-5-20251001",
    )


print("Contextual embedding functions defined.")
print()
print("NOTE: Building the full contextual index makes ~80 API calls (~$0.05).")
print("We'll demonstrate the concept with a small sample below instead.")

In [ ]:
# Demonstrate contextual embedding on a single chunk
# This shows how the description enriches the embedding without running the full ingest

bm25, chunks = load_bm25()

# Find the 2FA section specifically
twofa_chunk = next(
    (c for c in chunks if c['source'] == 'authentication.md' and 'two-factor' in c['section'].lower()),
    chunks[0]  # fallback to first chunk
)

print("=" * 65)
print("  Contextual Embedding: Single Chunk Demo")
print("=" * 65)
print()
print(f"Chunk: {twofa_chunk['source']} > {twofa_chunk['section']}")
print(f"\nOriginal text (first 300 chars):")
print(f"  {twofa_chunk['text'][:300]}...")

print()
print("Generating context description with Claude Haiku...")
context_desc = generate_context_description(
    twofa_chunk['text'], twofa_chunk['source'], twofa_chunk['section']
)

print(f"\nGenerated context description:")
print(f"  {context_desc}")

enriched_text = f"{context_desc}\n\n{twofa_chunk['text']}"
print(f"\nEnriched text (context + original):")
print(f"  {enriched_text[:400]}...")
print()
print("When we embed the enriched text, the description improves semantic alignment")
print("with queries like 'set up 2FA' or 'enable MFA' — which the original chunk missed.")

## Comparing All Three Approaches

For the query "How do I set up 2FA?", here's what each approach returns:

| Rank | Without Reranking | With Cross-Encoder | With LLM Reranker |
|------|---------------------------|--------------------------|---------------------------|
| #1 | authentication.md → **intro** | authentication.md → **2FA** | authentication.md → **2FA** |
| #2 | authentication.md → SSO | authentication.md → SSO | authentication.md → login |
| #3 | authentication.md → **2FA** | authentication.md → intro | authentication.md → SSO |

The 2FA section jumps from **#3 to #1** after reranking.

In [ ]:
# Trade-off comparison table
print("Trade-off Comparison: Three Reranking Approaches")
print("=" * 70)
print()

comparison = [
    {
        "approach": "No reranking",
        "latency": "~1s",
        "cost": "$0",
        "quality": "Poor (intro bias)",
        "domain_aware": "No",
    },
    {
        "approach": "Cross-encoder (local)",
        "latency": "~1.75s (+750ms)",
        "cost": "$0",
        "quality": "Good",
        "domain_aware": "Partial (web-trained)",
    },
    {
        "approach": "LLM reranker (Haiku)",
        "latency": "~4-6s (+3-5s)",
        "cost": "~$0.005/query",
        "quality": "Excellent",
        "domain_aware": "Yes (domain vocabulary)",
    },
    {
        "approach": "Contextual embeddings",
        "latency": "~1s (no query overhead)",
        "cost": "$0.05 once at ingest",
        "quality": "Good",
        "domain_aware": "Yes (at index time)",
    },
]

header = f"{'Approach':<26} {'Latency':<18} {'Cost':<16} {'Quality':<15} {'Domain-Aware'}"
print(header)
print("-" * len(header))
for row in comparison:
    print(f"{row['approach']:<26} {row['latency']:<18} {row['cost']:<16} {row['quality']:<15} {row['domain_aware']}")

print()
print("Recommendation:")
print("  Interactive chat with SaaS docs -> Cross-encoder (local, fast, free)")
print("  High-value queries, specialized domain -> LLM reranker (domain-aware)")
print("  System-wide quality improvement -> Contextual embeddings at ingest")

## Summary

**The key insight:**
> Retrieval gets the right **file**. Reranking gets the right **section** within the file.

The two-stage funnel:
1. **Hybrid retrieval** → cast a wide net (top 15 candidates, high recall)
2. **Reranking** → sort the net precisely (move best answer to #1, high precision)

**Real-world observation from our healthcare project:**
Contextual embeddings showed marginal improvement over hybrid search alone. The lesson: **fix retrieval first**, then optimize embeddings. The bottleneck was the semantic gap problem (Chapter 2), not embedding quality.

---

**Next:** [Chapter 4 — Complex Questions](../04-complex-questions/notebook.ipynb)

Even with perfect single-chunk retrieval, some questions require information from *multiple* sections across *multiple* files. Multi-query expansion, decomposition, and agentic RAG fix this.